# 06 — Asset Matching: Mock SOC Alerting Pipeline

The question a SOC actually asks when reading dark web content isn't *"what category is this page?"* — it's *"does anything on this page match my organization?"* This notebook builds that matcher.

## Approach

1. **Define a synthetic customer profile** — fake company, domain, employee emails, brand terms, executive names, known infrastructure.
2. **Inject ~40 seeded "hits"** into random CoDA pages — real emails replaced with customer emails, fake ransomware-leak posts naming the brand, a domain mention in a phishing-kit page, etc. This gives us *known ground truth* to measure against.
3. **Run the nb 05 DarkBERT entity extractor** on the corpus.
4. **Match extracted entities** against the customer profile — exact, normalized, and fuzzy string matching.
5. **Rank and emit alerts** — each match produces an analyst-facing JSON alert with severity, evidence snippet, and match type.
6. **Evaluate**: did we find the seeded hits? False-positive rate on the rest of the corpus?

## Why synthetic

Real dark-web-to-customer matching is an ethics minefield: actual leaks describe actual victims. Synthetic injection gives us known ground truth without exposing real orgs. The pipeline logic is identical to what a real dark-web-monitoring product runs — we just control the inputs.

## 1 — Synthetic customer profile

In [1]:
import json, random, re, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd

CUSTOMER = {
    'name':      'Acme Industrial Corp',
    'aliases':   ['Acme Industrial', 'AcmeCorp', 'Acme Corp'],
    'domains':   ['acme-industrial.com', 'acmeind.com', 'acme-corp.net'],
    'brands':    ['AcmePlant', 'AcmeOps', 'AcmeCloud'],
    'execs':     ['Alice Brennan', 'Vikram Sahni', 'Marcus Chen'],
    'employees_email_pattern': ['{first}.{last}@acme-industrial.com',
                                '{first}@acmeind.com'],
    # 25 synthetic employee emails
    'employee_emails': [],
    # known infrastructure (fake — for demonstration only)
    'known_ips':      ['203.0.113.42', '198.51.100.17'],
    'known_domains':  ['mail.acme-industrial.com', 'vpn.acme-industrial.com'],
}

_first = ['john', 'sarah', 'david', 'emily', 'michael', 'olivia', 'chris', 'maria',
          'robert', 'jennifer', 'daniel', 'lisa', 'james', 'patricia', 'william',
          'nancy', 'richard', 'betty', 'joseph', 'sandra', 'thomas', 'donna',
          'charles', 'carol', 'christopher']
_last = ['smith', 'johnson', 'brown', 'taylor', 'anderson', 'thomas', 'jackson',
         'white', 'harris', 'martin', 'thompson', 'garcia', 'martinez', 'robinson',
         'clark', 'rodriguez', 'lewis', 'lee', 'walker', 'hall', 'allen', 'young',
         'hernandez', 'king', 'wright']

random.seed(42)
emails = set()
while len(emails) < 25:
    f, l = random.choice(_first), random.choice(_last)
    pat = random.choice(CUSTOMER['employees_email_pattern'])
    emails.add(pat.format(first=f, last=l))
CUSTOMER['employee_emails'] = sorted(emails)

print(f'Customer: {CUSTOMER["name"]}')
print(f'  domains: {CUSTOMER["domains"]}')
print(f'  aliases: {CUSTOMER["aliases"]}')
print(f'  execs:   {CUSTOMER["execs"]}')
print(f'  25 employee emails generated (e.g., {CUSTOMER["employee_emails"][:3]})')

Customer: Acme Industrial Corp
  domains: ['acme-industrial.com', 'acmeind.com', 'acme-corp.net']
  aliases: ['Acme Industrial', 'AcmeCorp', 'Acme Corp']
  execs:   ['Alice Brennan', 'Vikram Sahni', 'Marcus Chen']
  25 employee emails generated (e.g., ['betty@acmeind.com', 'carol.harris@acme-industrial.com', 'charles.young@acme-industrial.com'])


## 2 — Load CoDA + inject seeded hits

40 injections covering different threat types. These are the *true positives* the matcher must find.

In [2]:
CODA_DIR = Path('data/coda/coda_dataset')
rows = []
for f in CODA_DIR.glob('*.txt'):
    parts = f.stem.split('-')
    if len(parts) < 4 or parts[2] != 'en' or parts[1] == 'Others':
        continue
    text = f.read_text(errors='ignore').strip()
    if len(text.split()) < 20:
        continue
    rows.append({'id': parts[0], 'category': parts[1], 'text': text})
coda = pd.DataFrame(rows).reset_index(drop=True)
print(f'Loaded {len(coda)} CoDA pages')

# --- Inject hits ----------------------------------------------------
random.seed(7)
INJECTION_RECIPES = [
    # (category_filter, templates, count) — pick pages matching category, insert template
    ('Financial', 'Fresh Acme Industrial Corp email dumps: {email} with working password. Contact for sample.', 10),
    ('Hacking',   'Selling RDP access to {domain} — admin panel, MFA bypassed. Price: 0.3 BTC.', 8),
    ('Hacking',   '*** LEAKED *** Acme Industrial Corp database — 240GB including customer records and HR data. Download mirrors below.', 5),
    ('Crypto',    'Executive targeting: {exec} from Acme Industrial Corp. LinkedIn profile scraped, social graph attached.', 4),
    ('Financial', 'Stealer log hit: {email}:Password123! saved from acme-industrial.com Outlook login.', 8),
    ('Hacking',   'New phish kit impersonates acme-industrial.com. Looking for customers, $300 setup fee.', 5),
]

injected = []  # list of (idx_in_coda, match_type, payload)
corpus = coda.copy()
for cat, tmpl, n in INJECTION_RECIPES:
    pool = corpus[corpus['category'] == cat].sample(min(n, len(corpus[corpus['category']==cat])),
                                                     random_state=random.randint(0, 1000)).index.tolist()
    for idx in pool:
        if '{email}' in tmpl:
            payload = tmpl.format(email=random.choice(CUSTOMER['employee_emails']))
            match_type = 'employee_email'
        elif '{domain}' in tmpl:
            payload = tmpl.format(domain=random.choice(CUSTOMER['domains']))
            match_type = 'domain'
        elif '{exec}' in tmpl:
            payload = tmpl.format(exec=random.choice(CUSTOMER['execs']))
            match_type = 'executive'
        else:
            payload = tmpl
            match_type = 'brand'
        # Prepend the injection so it lands in the first 512 tokens DarkBERT will see
        corpus.at[idx, 'text'] = payload + ' ' + corpus.at[idx, 'text']
        injected.append({'idx': idx, 'id': corpus.at[idx, 'id'],
                         'match_type': match_type, 'payload': payload})

inj_ids = {r['id'] for r in injected}
print(f'\nInjected {len(injected)} hits across {len(inj_ids)} pages')
print(f'Corpus size: {len(corpus)} (clean: {len(corpus) - len(inj_ids)}, seeded: {len(inj_ids)})')
print('\nInjection breakdown:')
print(pd.DataFrame(injected)['match_type'].value_counts().to_string())

Loaded 6582 CoDA pages

Injected 40 hits across 40 pages
Corpus size: 6582 (clean: 6542, seeded: 40)

Injection breakdown:
match_type
employee_email    18
brand             10
domain             8
executive          4


## 3 — Run the DarkBERT entity extractor from nb 05

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_PATH = Path('models/darkbert_entity_final')
assert MODEL_PATH.exists(), 'Run nb 05 first to produce the entity extractor.'

with open(MODEL_PATH / 'tag_vocab.json') as f:
    tag_vocab = json.load(f)
id2tag = {i: t for i, t in enumerate(tag_vocab['tags'])}

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_PATH))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()

MAX_LEN = 512
STRIDE = 64

def extract(text):
    """Return list of (char_start, char_end, label, surface_form) spans. Handles long pages via chunk+merge."""
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_offsets_mapping=True, return_overflowing_tokens=True,
                    stride=STRIDE, return_tensors='pt', padding=True).to(device)
    offsets = enc.pop('offset_mapping').cpu().numpy()
    enc.pop('overflow_to_sample_mapping', None)
    with torch.no_grad():
        logits = model(**enc).logits.cpu().numpy()
    tags = logits.argmax(-1)
    # Reassemble BIO spans per chunk, dedupe by (start, end)
    out = {}
    for chunk_tags, chunk_offs in zip(tags, offsets):
        cur = None
        for (s, e), tid in zip(chunk_offs, chunk_tags):
            if s == e == 0:
                continue
            tag = id2tag[int(tid)]
            if tag == 'O':
                if cur: out[(cur[0], cur[1])] = cur; cur = None
            elif tag.startswith('B-'):
                if cur: out[(cur[0], cur[1])] = cur
                cur = [int(s), int(e), tag[2:]]
            elif tag.startswith('I-') and cur and cur[2] == tag[2:]:
                cur[1] = int(e)
            else:
                if cur: out[(cur[0], cur[1])] = cur
                cur = [int(s), int(e), tag[2:]]
        if cur: out[(cur[0], cur[1])] = cur
    return [(s, e, l, text[s:e]) for (s, e), (_, _, l) in out.items()]

# Smoke test
demo = ("Acme Industrial Corp leak — contact john.smith@acme-industrial.com, wallet bc1qxy2kgdygjrsqtzq2n0yrf2493p83kkfjhx0wlh.")
print('Smoke-test extraction:')
for s, e, l, t in extract(demo):
    print(f'  {l:<18s} {t!r}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Smoke-test extraction:
  CRYPTO_WALLET      'bc1qxy2kgdygjrsqtzq2n0yrf2493p83kkfjhx0wlh'


In [4]:
# --- Run extractor on whole corpus -----------------------------------
EXTRACT_CACHE = Path('processed/coda_extractions.json')
EXTRACT_CACHE.parent.mkdir(exist_ok=True)

if EXTRACT_CACHE.exists():
    print('Loading cached extractions.')
    with open(EXTRACT_CACHE) as f:
        extractions = json.load(f)
else:
    print(f'Extracting entities from {len(corpus)} pages (a few min on GPU)...')
    extractions = {}
    for i, row in corpus.iterrows():
        spans = extract(row['text'])
        extractions[row['id']] = [[s, e, l, t] for s, e, l, t in spans]
        if (i + 1) % 500 == 0:
            print(f'  {i+1}/{len(corpus)}')
    with open(EXTRACT_CACHE, 'w') as f:
        json.dump(extractions, f)
    print(f'Cached to {EXTRACT_CACHE}')

total_ents = sum(len(v) for v in extractions.values())
print(f'Total entities extracted: {total_ents}')
label_counts = Counter(ent[2] for v in extractions.values() for ent in v)
print('By label:')
for l, c in label_counts.most_common():
    print(f'  {l:<18s} {c}')

Extracting entities from 6582 pages (a few min on GPU)...
  500/6582
  1000/6582
  1500/6582
  2000/6582
  2500/6582
  3000/6582
  3500/6582
  4000/6582
  4500/6582


KeyboardInterrupt: 

## 4 — Matching engine

Three match tiers, in order of confidence:

- **exact** — extracted entity equals a customer asset (case-insensitive, whitespace-normalized).
- **domain_contains** — extracted email/URL contains a customer domain.
- **fuzzy** — normalized edit-distance or substring match for aliases and brand terms (catches `Acme Corp` vs `AcmeCorp`).

We also scan the **raw text** (not just extracted entities) for brand/alias/exec mentions — brand keywords often don't form standalone entities.

In [ ]:
from difflib import SequenceMatcher

def normalize(s):
    return re.sub(r'\s+', '', s.strip().lower())

def fuzzy_ratio(a, b):
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

# Precompute lookup sets
cust_emails_norm = {e.lower() for e in CUSTOMER['employee_emails']}
cust_domains_norm = {d.lower() for d in CUSTOMER['domains']} | {d.lower() for d in CUSTOMER['known_domains']}
cust_ips = set(CUSTOMER['known_ips'])
cust_exec_tokens = {e.lower() for e in CUSTOMER['execs']}
cust_brand_tokens = {b.lower() for b in CUSTOMER['brands'] + CUSTOMER['aliases'] + [CUSTOMER['name']]}

def match_page(page_id, text, entities):
    """Return a list of match dicts for this page."""
    matches = []
    text_lo = text.lower()

    # --- entity-level matches ---
    for s, e, label, surface in entities:
        surf_lo = surface.lower()
        if label == 'EMAIL':
            if surf_lo in cust_emails_norm:
                matches.append({'type': 'employee_email', 'tier': 'exact',
                                'entity_label': label, 'surface': surface,
                                'evidence_span': (s, e)})
            else:
                # domain-contains fallback for emails at customer domains
                email_dom = surface.split('@')[-1].lower() if '@' in surface else ''
                if email_dom in cust_domains_norm:
                    matches.append({'type': 'domain_email', 'tier': 'domain_contains',
                                    'entity_label': label, 'surface': surface,
                                    'evidence_span': (s, e)})
        elif label == 'IP_ADDRESS':
            if surface in cust_ips:
                matches.append({'type': 'infrastructure_ip', 'tier': 'exact',
                                'entity_label': label, 'surface': surface,
                                'evidence_span': (s, e)})
        elif label == 'ORGANIZATION':
            r = max(fuzzy_ratio(surface, n) for n in [CUSTOMER['name']] + CUSTOMER['aliases'])
            if r >= 0.85:
                matches.append({'type': 'org_mention', 'tier': 'fuzzy',
                                'entity_label': label, 'surface': surface,
                                'evidence_span': (s, e), 'score': round(r, 3)})

    # --- raw-text scans for brand/domain/exec (may not form entities) ---
    for dom in cust_domains_norm:
        for m in re.finditer(re.escape(dom), text_lo):
            # skip if already counted via email match above
            if any(ms['evidence_span'][0] <= m.start() <= ms['evidence_span'][1]
                   for ms in matches if ms.get('entity_label') == 'EMAIL'):
                continue
            matches.append({'type': 'domain_mention', 'tier': 'exact',
                            'entity_label': None, 'surface': text[m.start():m.end()],
                            'evidence_span': (m.start(), m.end())})

    for tok in cust_brand_tokens:
        for m in re.finditer(r'\b' + re.escape(tok) + r'\b', text_lo):
            matches.append({'type': 'brand_mention', 'tier': 'exact',
                            'entity_label': None, 'surface': text[m.start():m.end()],
                            'evidence_span': (m.start(), m.end())})

    for exec_name in cust_exec_tokens:
        for m in re.finditer(r'\b' + re.escape(exec_name) + r'\b', text_lo):
            matches.append({'type': 'executive_mention', 'tier': 'exact',
                            'entity_label': None, 'surface': text[m.start():m.end()],
                            'evidence_span': (m.start(), m.end())})

    return matches

# --- apply to corpus -------------------------------------------------
alerts = []
for _, row in corpus.iterrows():
    ents = [tuple(e) for e in extractions.get(row['id'], [])]
    m = match_page(row['id'], row['text'], ents)
    for match in m:
        alerts.append({
            'page_id':    row['id'],
            'category':   row['category'],
            **match,
        })

alerts_df = pd.DataFrame(alerts)
print(f'Total match events: {len(alerts_df)}')
print(f'Pages with at least one match: {alerts_df["page_id"].nunique()}')
print('\nMatch types:')
print(alerts_df['type'].value_counts().to_string())

## 5 — Severity scoring + analyst-facing alerts

Roll up per-page match events into a single alert, with severity based on *what* matched (credential leak > brand mention) and *where* (Hacking/Financial categories weigh higher).

In [ ]:
SEVERITY_WEIGHTS = {
    'employee_email':    10,    # credential exposure — highest
    'infrastructure_ip':  9,
    'domain_email':       7,
    'org_mention':        6,
    'domain_mention':     5,
    'executive_mention':  5,
    'brand_mention':      3,
}
CATEGORY_WEIGHTS = {'Hacking': 1.5, 'Financial': 1.5, 'Crypto': 1.2}

def severity(group):
    base = max(SEVERITY_WEIGHTS.get(t, 1) for t in group['type'])
    cat_mult = CATEGORY_WEIGHTS.get(group['category'].iloc[0], 1.0)
    n_distinct = group['type'].nunique()
    return round(base * cat_mult + (n_distinct - 1) * 2, 2)

def snippet(text, span, window=120):
    s = max(0, span[0] - window); e = min(len(text), span[1] + window)
    return ('…' if s > 0 else '') + text[s:e].replace('\n', ' ') + ('…' if e < len(text) else '')

id2text = dict(zip(corpus['id'], corpus['text']))

rolled = []
for page_id, grp in alerts_df.groupby('page_id'):
    sev = severity(grp)
    # pick highest-severity match as the headline
    headline = grp.iloc[grp['type'].map(SEVERITY_WEIGHTS).fillna(0).argmax()]
    rolled.append({
        'page_id':    page_id,
        'category':   grp['category'].iloc[0],
        'severity':   sev,
        'match_count': len(grp),
        'match_types': sorted(grp['type'].unique().tolist()),
        'headline':   f'{headline["type"]}: {headline["surface"]}',
        'evidence':   snippet(id2text[page_id], headline['evidence_span']),
    })

alert_report = pd.DataFrame(rolled).sort_values('severity', ascending=False).reset_index(drop=True)
print(f'Distinct alerts: {len(alert_report)}')
print(f'\nTop 15 by severity:\n')
for _, r in alert_report.head(15).iterrows():
    print(f'[sev {r["severity"]:>5}] {r["category"]:<10s} {r["page_id"]}  {r["headline"]}')
    print(f'           evidence: {r["evidence"][:200]}')
    print()

## 6 — Evaluation against seeded ground truth

We know which pages were injected, and with what match type. Did the pipeline recover them? What's the false-positive rate on the other ~7k pages?

In [ ]:
seeded_ids = {r['id'] for r in injected}
flagged_ids = set(alert_report['page_id'])

tp = len(seeded_ids & flagged_ids)
fn = len(seeded_ids - flagged_ids)
fp = len(flagged_ids - seeded_ids)
tn = len(corpus) - len(seeded_ids) - fp

prec = tp / max(tp + fp, 1)
rec  = tp / max(tp + fn, 1)
f1   = 2 * prec * rec / max(prec + rec, 1e-9)

print(f'Seeded (positives):  {len(seeded_ids)}')
print(f'Flagged by pipeline: {len(flagged_ids)}')
print(f'')
print(f'  TP = {tp}   FN = {fn}')
print(f'  FP = {fp}   TN = {tn}')
print(f'')
print(f'  precision = {prec:.3f}   (of flagged, how many were real injections)')
print(f'  recall    = {rec:.3f}    (of injections, how many did we catch)')
print(f'  F1        = {f1:.3f}')

# --- Which injection types did we miss? ---
if fn:
    missed = [r for r in injected if r['id'] not in flagged_ids]
    print(f'\nMissed injections by type:')
    print(pd.DataFrame(missed)['match_type'].value_counts().to_string())

# --- Sample false positives ---
if fp:
    fp_sample = alert_report[~alert_report['page_id'].isin(seeded_ids)].head(5)
    print(f'\nSample false positives (pages flagged but never seeded):')
    for _, r in fp_sample.iterrows():
        print(f'  [sev {r["severity"]:>5}] {r["headline"]}')
        print(f'           {r["evidence"][:200]}')

## 7 — Save alerts as SOC-ready JSON

A real SOC pipeline emits one JSON blob per alert for downstream SIEM ingestion. Shape to match.

In [ ]:
def severity_band(s):
    if s >= 12: return 'critical'
    if s >= 8:  return 'high'
    if s >= 5:  return 'medium'
    return 'low'

out_alerts = []
for _, r in alert_report.iterrows():
    out_alerts.append({
        'alert_id':    hashlib.sha256(f'{r["page_id"]}-{r["headline"]}'.encode()).hexdigest()[:12],
        'source':      'darkweb-cti-pipeline',
        'customer':    CUSTOMER['name'],
        'severity':    severity_band(r['severity']),
        'severity_score': r['severity'],
        'page_id':     r['page_id'],
        'page_category': r['category'],
        'match_types': r['match_types'],
        'headline':    r['headline'],
        'evidence':    r['evidence'],
    })

OUT = Path('processed/soc_alerts.jsonl')
with open(OUT, 'w') as f:
    for a in out_alerts:
        f.write(json.dumps(a) + '\n')

print(f'Wrote {len(out_alerts)} alerts to {OUT}')
print(f'\nSeverity band distribution:')
print(pd.Series([a["severity"] for a in out_alerts]).value_counts().to_string())
print(f'\nSample alert:')
print(json.dumps(out_alerts[0], indent=2))

## What you've done

- Defined a realistic **customer asset profile** (domains, emails, execs, brands, infrastructure).
- Injected **known ground truth** into CoDA so the pipeline's accuracy can be measured.
- Chained the nb 05 extractor with a **three-tier matcher** (exact / domain-contains / fuzzy) plus a raw-text scan for brand terms that don't form entities.
- Rolled matches into **severity-scored alerts** that look like what a SOC SIEM would consume.
- Measured **recall on seeded injections and false-positive rate on the rest of the corpus.**

## Honest caveats

- **The seeded hits are prepended** to the page text, which means the extractor always sees them within the first 512 tokens. In real life injections would be buried mid-page — the recall number here is an upper bound.
- **Fuzzy matching is naive** (`difflib`). Production systems use normalized edit distance with hand-tuned thresholds per match type, or embedding similarity.
- **No entity linking.** If a page mentions "Acme Corp" but means a *different* Acme, we have no disambiguation. Real systems add corroborating evidence (industry, geography, product names) before alerting.

## What's next

- **Nb 07 — Ransomware leak-site detection.** Using DarkBERT's bundled `leaksite_detection_dataset.tsv`, train a binary classifier that flags "this post is a ransomware victim announcement."
- **Nb 08 — Credential dump parser.** Synthetic stealer logs + HIBP breach metadata. The credential-leak side of the SOC pipeline.